In [0]:
SELECT
  COUNT(*) AS total_itens_vendidos,
  CAST(ROUND(SUM(preco_venda),2) AS DECIMAL(10,2)) AS receita_total,
  COUNT(DISTINCT id_pedido) AS total_pedidos,
  CAST(ROUND(SUM(preco_venda)/COUNT(DISTINCT id_pedido),2) AS DECIMAL(10,2)) AS ticket_medio_geral,
  COUNT(DISTINCT id_cliente) AS clientes_unicos,
  ROUND(COUNT(DISTINCT id_pedido)/COUNT(DISTINCT id_cliente),2) AS pedidos_por_cliente
FROM ecommerce_mvp.gold.fato_vendas;


In [0]:
SELECT date_format(t.mes, 'MM/yyyy') AS mes, receita_total, qtd_pedidos, ticket_medio, clientes_ativos, frequencia_media
FROM ecommerce_mvp.gold.kpis_mensais t
ORDER BY t.mes DESC

In [0]:
--Por dia da semana
SELECT date_format(data_venda, 'EEEE') AS dia_semana,
       CAST(ROUND(SUM(preco_venda),2) AS DECIMAL(10,2)) AS receita,
       COUNT(DISTINCT id_pedido) AS pedidos
FROM ecommerce_mvp.gold.fato_vendas
GROUP BY date_format(data_venda, 'EEEE')
ORDER BY receita DESC

In [0]:
-- Tendência mensal (últimos 15 meses)
SELECT date_format(t.mes, 'MM/yyyy') AS mes, receita_total, qtd_pedidos, ticket_medio, clientes_ativos, frequencia_media
FROM ecommerce_mvp.gold.kpis_mensais t
ORDER BY t.mes DESC LIMIT 15;

In [0]:
--Taxa geral
SELECT status, COUNT(*) AS qtd,
       ROUND(COUNT(*)*100.0/SUM(COUNT(*)) OVER(),1) AS pct
FROM ecommerce_mvp.silver.pedidos
GROUP BY status ORDER BY qtd DESC;

In [0]:
-- Por categoria de produto
SELECT p.categoria,
       COUNT(*) AS total_itens,
       SUM(CASE WHEN ped.status IN ('Cancelled','Returned') THEN 1 ELSE 0 END) AS cancel_devol,
       ROUND(SUM(CASE WHEN ped.status IN ('Cancelled','Returned') THEN 1 ELSE 0 END)*100.0/COUNT(*),1) AS pct_cancel_devol
FROM ecommerce_mvp.silver.itens_pedido ip
JOIN ecommerce_mvp.silver.produtos p ON ip.id_produto = p.id_produto
JOIN ecommerce_mvp.silver.pedidos ped ON ip.id_pedido = ped.id_pedido
GROUP BY p.categoria
ORDER BY pct_cancel_devol DESC;

In [0]:
--Categorias mais vendidas
SELECT categoria, SUM(unidades_vendidas_total) AS unidades,
       CAST(ROUND(SUM(receita_total),2) AS DECIMAL(10,2)) AS receita
FROM ecommerce_mvp.gold.giro_vendas_produto
GROUP BY categoria
ORDER BY receita DESC

In [0]:
--Margem por categoria
SELECT categoria,
       COUNT(*) AS unidades,
       ROUND(SUM(margem),2) AS margem_total,
       ROUND(AVG(margem),2) AS margem_media_unidade,
       ROUND(SUM(margem)*100.0/SUM(preco_venda),1) AS pct_margem
FROM ecommerce_mvp.gold.fato_vendas
GROUP BY categoria
ORDER BY pct_margem DESC;


In [0]:
--Departamento
SELECT departamento,
       COUNT(*) AS unidades,
       ROUND(SUM(preco_venda),2) AS receita,
       ROUND(SUM(margem),2) AS margem_total,
       ROUND(SUM(margem)*100.0/SUM(preco_venda),1) AS pct_margem
FROM ecommerce_mvp.gold.fato_vendas
GROUP BY departamento
ORDER BY receita DESC;


In [0]:
--Estoque
SELECT g.id_produto, p.nome_produto, g.categoria,
       g.unidades_vendidas_90d, COALESCE(e.unidades_em_estoque, 0) AS estoque_atual,
       CASE WHEN COALESCE(e.unidades_em_estoque,0) = 0 THEN NULL
            ELSE ROUND(g.unidades_vendidas_90d / e.unidades_em_estoque, 2) END AS indice_giro
FROM ecommerce_mvp.gold.giro_vendas_produto g
LEFT JOIN ecommerce_mvp.gold.estoque_atual e ON g.id_produto = e.id_produto
JOIN ecommerce_mvp.gold.dim_produto p ON g.id_produto = p.id_produto
ORDER BY g.unidades_vendidas_90d DESC
LIMIT 20


In [0]:

--Estoque parado
SELECT COUNT(*) AS produtos_com_estoque,
       SUM(CASE WHEN dias_medios_parado > 90 THEN 1 ELSE 0 END) AS produtos_parados_90d,
       ROUND(AVG(dias_medios_parado),1) AS media_geral_dias
FROM ecommerce_mvp.gold.estoque_atual;


In [0]:
--Estoque parado por produto
SELECT e.id_produto, p.nome_produto, e.unidades_em_estoque, e.dias_medios_parado
FROM ecommerce_mvp.gold.estoque_atual e
JOIN ecommerce_mvp.gold.dim_produto p ON e.id_produto = p.id_produto
WHERE e.dias_medios_parado > 90
ORDER BY e.dias_medios_parado DESC;

In [0]:

--Tempo de entrega
SELECT ROUND(AVG(DATEDIFF(data_entrega, data_pedido)),1) AS dias_medios_entrega,
       COUNT(*) AS pedidos_entregues
FROM ecommerce_mvp.silver.pedidos
WHERE data_entrega IS NOT NULL;

SELECT cd.nome AS centro_distribuicao,
       COUNT(DISTINCT ped.id_pedido) AS pedidos,
       ROUND(AVG(DATEDIFF(ped.data_entrega, ped.data_pedido)),1) AS dias_medios_entrega
FROM ecommerce_mvp.silver.pedidos ped
JOIN ecommerce_mvp.silver.itens_pedido ip ON ped.id_pedido = ip.id_pedido
JOIN ecommerce_mvp.silver.produtos p ON ip.id_produto = p.id_produto
JOIN ecommerce_mvp.silver.centros_distribuicao cd ON p.id_centro_distribuicao = cd.id_centro_distribuicao
WHERE ped.data_entrega IS NOT NULL
GROUP BY cd.nome
ORDER BY dias_medios_entrega DESC;


In [0]:
--Devolução por centro de distribuição
SELECT cd.nome AS centro_distribuicao,
       COUNT(DISTINCT ped.id_pedido) AS pedidos,
       ROUND(SUM(CASE WHEN ped.status IN ('Cancelled','Returned') THEN 1 ELSE 0 END)*100.0
             / COUNT(DISTINCT ped.id_pedido), 1) AS pct_cancel_devol
FROM ecommerce_mvp.silver.pedidos ped
JOIN ecommerce_mvp.silver.itens_pedido ip ON ped.id_pedido = ip.id_pedido
JOIN ecommerce_mvp.silver.produtos p ON ip.id_produto = p.id_produto
JOIN ecommerce_mvp.silver.centros_distribuicao cd ON p.id_centro_distribuicao = cd.id_centro_distribuicao
GROUP BY cd.nome
ORDER BY pct_cancel_devol DESC;


In [0]:
--ciclo de vida
SELECT ROUND(AVG(MONTHS_BETWEEN(mes, mes_anterior)), 1) AS meses_medios_ate_recompra
FROM (
    SELECT id_cliente, mes,
           LAG(mes) OVER (PARTITION BY id_cliente ORDER BY mes) AS mes_anterior
    FROM (SELECT DISTINCT id_cliente, DATE_TRUNC('month', data_venda) AS mes FROM ecommerce_mvp.gold.fato_vendas)
) t
WHERE mes_anterior IS NOT NULL;

In [0]:
--novs, retidos e reativados
SELECT mes, classificacao, count(*) AS qtd_clientes
FROM ecommerce_mvp.gold.ciclo_vida_cliente
GROUP BY mes, classificacao
ORDER BY mes;

-- Distribuição percentual agregada (todos os meses)
SELECT classificacao, count(*) AS qtd,
       ROUND(count(*)*100.0/SUM(count(*)) OVER(),1) AS pct
FROM ecommerce_mvp.gold.ciclo_vida_cliente
GROUP BY classificacao
ORDER BY qtd DESC;


In [0]:
--base de clientes
SELECT segmento_crm, count(*) AS qtd_clientes, ROUND(AVG(receita_total_cliente),2) AS receita_media
FROM ecommerce_mvp.gold.base_crm
GROUP BY segmento_crm;


In [0]:

-- lista de clientes para o segmento "Alvo de reativação"
SELECT * FROM ecommerce_mvp.gold.base_crm WHERE segmento_crm = 'Alvo de reativação';

In [0]:
--Canal de origem
SELECT c.origem_trafego,
       COUNT(DISTINCT c.id_cliente) AS clientes,
       ROUND(COUNT(DISTINCT c.id_cliente)*100.0/SUM(COUNT(DISTINCT c.id_cliente)) OVER(),1) AS pct_clientes,
       ROUND(SUM(v.preco_venda),2) AS receita_total,
       ROUND(SUM(v.preco_venda)/COUNT(DISTINCT c.id_cliente),2) AS receita_media_por_cliente
FROM ecommerce_mvp.gold.dim_cliente c
LEFT JOIN ecommerce_mvp.gold.fato_vendas v ON c.id_cliente = v.id_cliente
GROUP BY c.origem_trafego
ORDER BY clientes DESC;


In [0]:
--distribuição geográfica de receita
SELECT c.pais, COUNT(DISTINCT v.id_cliente) AS clientes, ROUND(SUM(v.preco_venda),2) AS receita
FROM ecommerce_mvp.gold.fato_vendas v
JOIN ecommerce_mvp.gold.dim_cliente c ON v.id_cliente = c.id_cliente
GROUP BY c.pais
ORDER BY receita DESC;


In [0]:
--por genêro
SELECT genero, categoria, unidades
FROM (
    SELECT c.genero, v.categoria, COUNT(*) AS unidades,
           ROW_NUMBER() OVER (PARTITION BY c.genero ORDER BY COUNT(*) DESC) AS rn
    FROM ecommerce_mvp.gold.fato_vendas v
    JOIN ecommerce_mvp.gold.dim_cliente c ON v.id_cliente = c.id_cliente
    GROUP BY c.genero, v.categoria
) ranked
WHERE rn <= 5
ORDER BY genero, unidades DESC;

In [0]:
-- Por faixa etária x departamento
SELECT
  CASE WHEN c.idade < 25 THEN '<25' WHEN c.idade < 40 THEN '25-39'
       WHEN c.idade < 55 THEN '40-54' ELSE '55+' END AS faixa_etaria,
  v.departamento, COUNT(*) AS unidades
FROM ecommerce_mvp.gold.fato_vendas v
JOIN ecommerce_mvp.gold.dim_cliente c ON v.id_cliente = c.id_cliente
GROUP BY faixa_etaria, v.departamento
ORDER BY faixa_etaria, v.departamento;

In [0]:
--Funil
SELECT
    COUNT(*) AS total_sessoes,
    SUM(viu_produto) AS sessoes_viram_produto,
    SUM(adicionou_carrinho) AS sessoes_add_carrinho,
    SUM(comprou) AS sessoes_compraram,
    ROUND(SUM(adicionou_carrinho) / NULLIF(SUM(viu_produto), 0) * 100, 1) AS pct_produto_para_carrinho,
    ROUND(SUM(comprou) / NULLIF(SUM(adicionou_carrinho), 0) * 100, 1) AS pct_carrinho_para_compra,
    ROUND(SUM(comprou) / NULLIF(SUM(viu_produto), 0) * 100, 1) AS pct_produto_para_compra
FROM ecommerce_mvp.gold.funil_conversao;


In [0]:
SELECT
    origem_trafego,
    COUNT(*) AS total_sessoes,
    SUM(viu_produto) AS sessoes_viram_produto,
    SUM(comprou) AS sessoes_compraram,
    ROUND(SUM(comprou) / NULLIF(SUM(viu_produto), 0) * 100, 1) AS pct_conversao
FROM ecommerce_mvp.gold.funil_conversao
GROUP BY origem_trafego
ORDER BY pct_conversao DESC;
